# EmpowerLens — model-triage-label experiment (Kaggle GPU runner)

Runs the real transformer pipeline on **model-triage-predicted labels**
(`Annotated_data_model_triage.csv` — Dominant/Secondary Distortion filled
in from `model_predicted_primary`/`model_predicted_secondary` in
`triage_full.csv`, 0.3-prob cutoff for `No Distortion`) instead of the
real hand-annotated labels. This is pass 1 of a 2-pass comparison: pass 2
will re-run this same notebook against the manually corrected CSV once
the `audit_queue.csv` review is done, so the two `results_*` folders can
be diffed directly.

**Before running:**
1. Settings → **Accelerator: GPU**, **Internet: On**.
2. Attach `Annotated_data_model_triage.csv` as a Kaggle **Dataset** input
   (Add Data → Upload) and set `TRIAGE_CSV_INPUT` below to its path under
   `/kaggle/input/...`.
3. Splits here are generated **fresh inside this notebook**, into
   `data/splits_model_triage/`, NOT the frozen official `data/splits/` —
   this is a one-off comparison run on non-frozen labels, so it doesn't
   touch the committed splits at all.

In [ ]:
# 1. Clone the repo and install the transformer stack.
from kaggle_secrets import UserSecretsClient
import os
from huggingface_hub import login

# Pulls securely from your Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

# Path to the uploaded CSV once you've attached it as a Kaggle Dataset input.
TRIAGE_CSV_INPUT = "/kaggle/input/datasets/lumiaqureshi/annotated-model-triage/Annotated_data_model_triage.csv"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
!cp "$TRIAGE_CSV_INPUT" ./Annotated_data_model_triage.csv
!ls -la Annotated_data_model_triage.csv

In [ ]:
# 2. Generate FRESH splits for the model-triage labels (never touches the
#    frozen data/splits/). Same 80/10/10 multi-label stratified method,
#    seed=42, just pointed at a different source CSV and output dir.
!python -m src.make_splits --path Annotated_data_model_triage.csv \
    --out data/splits_model_triage --seed 42 

In [ ]:
# 3. Choose ONE task, train it over the three seeds, evaluate each on
#    val+test, then aggregate. Everything reads/writes the *_model_triage
#    dirs so this run never overwrites the official results/.
TASK = "multiclass"  # one of: binary | multiclass | multilabel
MODEL = "mental/mental-roberta-base"  # matches oof_meta.json

for seed in (42, 1337, 2024):
    ckpt = f"checkpoints_model_triage/{TASK}_{MODEL.split(chr(47))[-1]}_{seed}"
    !python -m src.train_transformer --task $TASK --model $MODEL --seed $seed --device auto \
        --splits data/splits_model_triage --out checkpoints_model_triage
    !python -m src.evaluate --checkpoint $ckpt --reference \
        --splits data/splits_model_triage --out results_model_triage

!python -m src.aggregate --results results_model_triage

In [ ]:
# 4. Copy results_model_triage/ to the Kaggle output so it can be
#    downloaded from the session (and diffed later against the
#    corrected-labels run of the same notebook).
!mkdir -p /kaggle/working/results_model_triage
!cp -r results_model_triage/* /kaggle/working/results_model_triage/
!ls -la /kaggle/working/results_model_triage